# Q-Learning in Frozen Lake and Taxi-v3

Quality learning is an **off-policy**, **value-based** method that uses **Temporal Difference** approach to train its action-value function, udating it at each step intead at the end of the eopisode, using immediate reward plus the discounted value of the next state, computed by finding the action that *maximizes the current Q-function at the next state*, bootstraping. The set of all state-action pairs, Q-table, number of columns = possible actions, number of rows =  posible states is used and updated. Given a state and action, the **Q-function** will search the Q-table to output the (state-action) pair value, the action chosen using **epsilon-greedy** strategy=**action policy**:, **exploration**=random action with probability epsilon, and **exploitation** with probability 1-epsilon, start with large epislon,  as training progresses, Q-table gets better in its estimation, and espolon value is reduced. Traiining achives optimal Q-function, thus optimal policy which is the best action to take at each state, which is defferent than the **updating policy**, which is used dusing training to update the Q-value.

Q-learning is a simple algorithm designed for a smaller and discrete environment

## Install dependencies

In [ ]:
!pip install gymnasium pygame numpy huggingface_hub pyyaml imageio imageio_ffmpeg pyglet tqdm

## Create a virtual display

In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg xvfb
!pip3 install pyvirtualdisplay

In [ ]:
# restart notebook runtime
import os
os.kill(os.getpid(), 9)

In [ ]:
# create virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd drive/MyDrive/ColabNotebooks/DeepRL/q_learning

Mounted at /content/drive
/content/drive/MyDrive/ColabNotebooks/DeepRL/q_learning


## Import Packages

In [ ]:
import os
import tqdm
import random
import imageio

import numpy as np
import gymnasium as gym

import pickle as pickle

from tqdm.notebook import tqdm

## <font size="20">**Frozen Lake**</font>

Train Q-Learning agent to navidate from the starting state to the goal state by walking only on frozen tiles F and avoid holes. The environment mode is non-slippery 4x4 map, setting `render_mode` to `rgb_array`  which returns a single frame, np.ndarray (x, y, 3) representing RGB values of x-by-y image of the current state of the environment. Gymnasium.name() creates a reinforcement learning environment by looking environmet unique name and version finding the environment in an internal registry, loading and instantiating it

### Environment

In [ ]:
# create environment, the agent moves in the intended direction due to non-slippery frozen lake (deterministic)
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")

In [ ]:
# create custom grid or use default environment
desc = ["SFFF", "FHFH", "FFFH", "HFFG"]
custom_env = gym.make("FrozenLake-v1", desc=desc, is_slippery=True)

`env.observation_space` is an integer representing the agent's current position as current row x number of columns + current column, where both raw and columnst start at 0

In [ ]:
# display the environment
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space", env.observation_space)
print("Sample observation", env.observation_space.sample()) # random observation

_____OBSERVATION SPACE_____ 

Observation Space Discrete(16)
Sample observation 3


The action space is the set of possible actions the agent can take:

* 0: GO LEFT
* 1: GO DOWN
* 2: GO RIGHT
* 3: GO UP

Reward function:

* Reach the goal: +1
* Falling into the hole: 0
* Stayung on the frozen lake: 0

In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action


 _____ACTION SPACE_____ 

Action Space Shape 4
Action Space Sample 2


In [ ]:
state_space = env.observation_space.n
action_space = env.action_space.n

print("There are ", state_space, " possible states")
print("There are ", action_space, " possible actions")

There are  16  possible states
There are  4  possible actions


## Q-table

Init Q-table with number of raws = states: start, staing on the frozen lake, hole, end, and number of columns = actions: right, left, up, down

In [ ]:
# init Q-table of size state_space x action_space with each value at 0
def initialize_q_table(state_space, action_space):
  Qtable = np.zeros((state_space, action_space))
  return Qtable

In [ ]:
Qtable_frozenlake = initialize_q_table(state_space, action_space)
# display Qtable
Qtable_frozenlake

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### Greedy Policy

Q-Learning is an **off-policy** algorithm using a different policies for acting and updating the state-action pair values:
* Epsilon-greedy policy for acting
* Greedy-policy for updating, also the final policy, used to select an action using Q-table

In [ ]:
def greedy_policy(Qtable, state):
  """Exploatation, takes the action with the highest state action value """
  # Qtable[state][:]
  print(f"The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:\n {Qtable[state][:]}""")
  # finds the index of the action with the highest Q-value in the current state's
  action = np.argmax(Qtable[state][:])
  return action

### Epsilon-greedy policy

Espilon-greedy is training policy that handles the exploration/ explaitiation trade-off:

* **exploration** a random action with probability ɛ
* **exploatation** with proability 1 - ɛ

The ɛ value is reduced with training

In [ ]:
def epsilon_gredy_policy(Qtable, state, epsilon):
  """ Based on a randomly generated naumber, take an exploration or explotationaction"""
  random_num = random.uniform(0,1)
  if random_num > epsilon:
    # exploatation = take the action with the highest value with a given state
    action = greedy_policy(Qtable, state)
  else:
    # exloaration = take a random action
    action = env.action_space.sample()
  return action

## Hyperparameters

In [ ]:
# training parameters
n_training_episodes = 10000  # total training episodes
learning_rate = 0.7          # learning rate

# evaluation parameters
n_eval_episodes = 100        # total number of test episodes

# environment parameters
env_id = "FrozenLake-v1"     # name of the environment
max_steps = 99               # max steps per episode
gamma = 0.95                 # discounting rate
eval_seed = []               # evaluation seed of the environment

# exploration parameters - decrease epsilon swoly in order to explore enough of the state space
max_epsilon = 1.0             # exploration probability at start
min_epsilon = 0.05            # minimum exploration probability
decay_rate = 0.0005           # exponential decay rate for exploration prob

## Training

For episode in the total training episodes:

Reduce epsilon
Reset environment

  For step in max timesteps:
    Choose action At using epsilon greedy policy
    Take action and observe outcome state s' and reward r.
    Update Q-value Q(s, a) using Bellman equation Q(s, a) + lr[R(s, a) + gamma * max Q(s', a') - Q(s, a)]

In [ ]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  """ Defines the Loop for training episodes. After completing all of the training episodes
      the function will return the updated Q-Table. """
  for episode in tqdm(range(n_training_episodes)):
    # reduce epsilon for less exploration: as the episode number increases the exponential decay factor decreases
    # causing epsilon to decay
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # reset environment
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    # a nested loop for the maximum steps
    for step in range(max_steps):
      # choose action using epsilon greedy policy
      action = epsilon_gredy_policy(Qtable, state, epsilon)
      # take the action At, and observe the outcome state St+1 and the reward Rt+1
      new_state, reward, terminated, truncated, info = env.step(action)
      # update Q table using Bellman equation Q(s, a) := Q(s, a) + lr[R(s, a) + gamma * max Q(s', a') - Q(s, a)]
      Qtable[state][action] = Qtable[state][action] + learning_rate * (reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action])
      # if terminated or truncated finish the episode
      if terminated or truncated:
        break
      # update state
      state = new_state
  return Qtable

In [ ]:
# train Q-learning agent generates the values in the Q-Table, which the agent will use to navigate the environment and achive the goal
Qtable_frozenlake = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake)

  0%|          | 0/10000 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.9025 0.95   1.     0.9025]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.73509189 0.77378094 0.77378094 0.73509189]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.77378094 0.81450625 0.         0.73509189]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.81450625 0.         0.857375   0.77378094]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.81450625 0.9025     0.9025     0.        ]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.

In [ ]:
# display the learned Q-table
Qtable_frozenlake

array([[0.73509189, 0.77378094, 0.77378094, 0.73509189],
       [0.73509189, 0.        , 0.81450625, 0.77378094],
       [0.77378094, 0.857375  , 0.77378094, 0.81450625],
       [0.81450625, 0.        , 0.77378094, 0.77378094],
       [0.77378094, 0.81450625, 0.        , 0.73509189],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.9025    , 0.        , 0.81450625],
       [0.        , 0.        , 0.        , 0.        ],
       [0.81450625, 0.        , 0.857375  , 0.77378094],
       [0.81450625, 0.9025    , 0.9025    , 0.        ],
       [0.857375  , 0.95      , 0.        , 0.857375  ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.9025    , 0.95      , 0.857375  ],
       [0.9025    , 0.95      , 1.        , 0.9025    ],
       [0.        , 0.        , 0.        , 0.        ]])

## Evaluation

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
  """
  Evaluate the agent for ``n_eval_episodes`` and returns average reward and std of reward.
  :param env: evaluation environment
  :param max_steps: maximum number of timesteps per episode
  :param n_eval_episodes: number of episode to evaluate the agent
  :param Q: Q-table
  :param seed: eval.uation seed array for taxi-v3
  """
  episode_rewards = []

  for episode in tqdm(range(n_eval_episodes)):
    if seed:
      # reset the environment with seed
      state, info = env.reset(seed=seed[episode])
    else:
      # reset the environment without a seed
      state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards = 0

    # in a nested loo the agent will run to max_steps, and take the action that
    # maximizes the expected future reward in a given state using Q-Table
    for step in range(max_steps):
      # based on the learnd Q-table choose the index for the action that have the max expected future reward for the given state
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      # calculate the reward
      total_rewards += reward

      if terminated or truncated:
        break
      # change the state
      state = new_state
    # append the results
    episode_rewards.append(total_rewards)
  # use the accumilated results to caclulate the mean and std
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

In [ ]:
# evaluate Q-learning agent
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake, eval_seed)

  0%|          | 0/100 [00:00<?, ?it/s]

The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.73509189 0.77378094 0.77378094 0.73509189]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.77378094 0.81450625 0.         0.73509189]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.81450625 0.         0.857375   0.77378094]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.81450625 0.9025     0.9025     0.        ]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.       0.9025   0.95     0.857375]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [0.9025 0.95   1.     0.9025]
The row of Qtabl

In [ ]:
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

Mean_reward=1.00 +/- 0.00


## Publish trained model

In [ ]:
import os
import base64                                 # encoding binary data into ASCII characters

from pathlib import Path                      # to interact with file system paths
from IPython import display as ipythondisplay # displays varius output types including html

from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save
from huggingface_hub import notebook_login

from pathlib import Path
import datetime
import json

notebook_login()

In [ ]:
os.system("Xvfb :1 -screen 0 1024x768x24 &")  # start a virutal display with shell command, a X virtual frame buffer
os.environ['DISPLAY'] = ':1'

In [ ]:
def record_video(env, Qtable, out_directory, fps=1):
  """
  Generate a replay video of the agent from the start till reaching the goal
  :param env
  :param Qtable: Qtable of our agent
  :param out_directory
  :param fps: how many frame per seconds (with taxi-v3 and frozenlake-v1 we use 1)
  """
  images = []
  terminated = False
  truncated = False
  # create a state by resetting the environment with a random integer bw 0 and 500
  state, info = env.reset(seed=random.randint(0,500))
  # render the environment
  img = env.render()
  # appemd the img to the images array
  images.append(img)
  # loop of taking steps using the Q-Table and redering the image for every step
  while not terminated or truncated:
    # choose the action(index) that have the maximum expected future reward given that state
    action = np.argmax(Qtable[state][:])
    state, reward, terminated, truncated, info = env.step(action)
    img = env.render()
    images.append(img)
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [ ]:
def show_videos(video_path="", prefix=""):
    """
    Display video from folder
    :param video_path: (str) Path to the folder containing videos
    :param prefix: (str) Filter the video, showing only the only starting with this prefix
    """
    html = []
    # find all .mp4 files in specified path that match prefix
    for mp4 in Path(video_path).glob("{}*.mp4".format(prefix)):
        # the binary content of video file is read and encoded into base64 string in order to be embeded in HTML output
        video_b64 = base64.b64encode(mp4.read_bytes())
        # append an html tag, configured to autoplay, loop and display controls
        html.append(
            """<video alt="{}" autoplay
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{}" type="video/mp4" />
                </video>""".format(
                mp4, video_b64.decode("ascii")
            )
        )
    # the list of html video tags are joined with line brake and displays the resulting html
    ipythondisplay.display(ipythondisplay.HTML(data="<br>".join(html)))

In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_frozenlake
}

In [ ]:
model["qtable"]

array([[0.73509189, 0.77378094, 0.77378094, 0.73509189],
       [0.73509189, 0.        , 0.81450625, 0.77378094],
       [0.77378094, 0.857375  , 0.77378094, 0.81450625],
       [0.81450625, 0.        , 0.77378094, 0.77378094],
       [0.77378094, 0.81450625, 0.        , 0.73509189],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.9025    , 0.        , 0.81450625],
       [0.        , 0.        , 0.        , 0.        ],
       [0.81450625, 0.        , 0.857375  , 0.77378094],
       [0.81450625, 0.9025    , 0.9025    , 0.        ],
       [0.857375  , 0.95      , 0.        , 0.857375  ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.9025    , 0.95      , 0.857375  ],
       [0.9025    , 0.95      , 1.        , 0.9025    ],
       [0.        , 0.        , 0.        , 0.        ]])

In [ ]:
record_video(env, model["qtable"], "videos/frl-replay.mp4")

In [ ]:
show_videos("videos", prefix="frl-")

In [ ]:
def push_to_hub(repo_id, model, env, video_fps=1, local_repo_path="hub"):
  """ This method performs the complete pipeline:
  - evaluates the model
  - generates the model card
  - makes a replay video of the agent
  - it pushes all of the above to the hub

  :param repo_id: the model repository at HuggingFace Hub
  :param env
  :param video_fps: how many frames per second to record for video replay
  :param local_repo_path: path to local repository
  """

  _, repo_name = repo_id.split("/")

  eval_env = env
  api = HfApi()

  # create the repo
  repo_url = api.create_repo(repo_id=repo_id, exist_ok=True)

  # download files
  repo_local_path = Path(snapshot_download(repo_id=repo_id))

  # save model
  if env.spec.kwargs.get("map_name"):
    model["map_name"] = env.spec.kwargs.get("map_name")
    if env.spec.kwargs.get("is_slippery", "") == False:
      model["slippery"] = False

  # pickle the model
  with open((repo_local_path)/"q-learning.pkl", "wb") as f:
    pickle.dump(model, f)

  # evaluate the model
  mean_reward, std_reward = evaluate_agent(
      eval_env,
      model["max_steps"],
      model["n_eval_episodes"],
      model["qtable"],
      model["eval_seed"]
      )
  evaluate_data = {
      "env_id": model["env_id"],
      "mean_reward": mean_reward,
      "n_eval_episodes": model["n_eval_episodes"],
      "eval_datetime": datetime.datetime.now().isoformat()
  }

  # write the evaluation results into a json file
  with open(repo_local_path / "results.json", "w") as outfile:
    json.dump(evaluate_data, outfile)

  # create model card
  env_name = model["env_id"]
  if env.spec.kwargs.get("map_name"):
    env_name += "-" + env.spec.kwargs.get("map_name")

  if env.spec.kwargs.get("is_slippery", "") == False:
    env_name += "-" + "no_slippery"

  metadata = {}
  metadata["tags"] = [env_name, "q-learning", "reinforcement-learning", "custom-implementation"]

  # add metrics
  eval = metadata_eval_result(
      model_pretty_name=repo_name,
      task_pretty_name="reinforcement-learning",
      task_id="reinforcement-learning",
      metrics_pretty_name="mean_reward",
      metrics_id="mean_reward",
      metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
      dataset_pretty_name=env_name,
      dataset_id=env_name
  )

  # merge dictionaries
  metadata = {**metadata, **eval}
  model_card = f"""
  # **Q-Learning** Agent playing1 **{env_id}**
  This is a trained model of a **Q-Learning** agent playing **{env_id}**

  ## Usage

  ```python

  model = load_from_hub(repo_id="{repo_id}", filename="q-learning.pkl")

  # check if additional attributes like is_slippery=False etc are needed
  env = gym.make(model["env_id"])
  ```
  """

  evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

  readme_path = repo_local_path / "README.md"
  readme = ""
  print(readme_path.exists())
  if readme_path.exists():
      with readme_path.open("r", encoding="utf8") as f:
          readme = f.read()
  else:
      readme = model_card

  with readme_path.open("w", encoding="utf-8") as f:
      f.write(readme)

  # save metrics to Readme metadata
  metadata_save(readme_path, metadata)

  # record a video
  video_path = repo_local_path / "replay.mp4"
  record_video(env, model["qtable"], video_path, video_fps)

  # push everything to the Hub
  api.upload_folder(
      repo_id=repo_id,
      folder_path=repo_local_path,
      path_in_repo=".",
  )

  print("Model is pushed to the Hub. View model at: ", repo_url)

Create a model dictionary that contains the hyperparameters and Q-table

In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_frozenlake
}

In [ ]:
model

{'env_id': 'FrozenLake-v1',
 'max_steps': 99,
 'n_training_episodes': 10000,
 'n_eval_episodes': 100,
 'eval_seed': [],
 'learning_rate': 0.7,
 'gamma': 0.95,
 'max_epsilon': 1.0,
 'min_epsilon': 0.05,
 'decay_rate': 0.0005,
 'qtable': array([[0.73509189, 0.77378094, 0.77378094, 0.73509189],
        [0.73509189, 0.        , 0.81450625, 0.77378094],
        [0.77378094, 0.857375  , 0.77378094, 0.81450625],
        [0.81450625, 0.        , 0.77378094, 0.77378094],
        [0.77378094, 0.81450625, 0.        , 0.73509189],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.9025    , 0.        , 0.81450625],
        [0.        , 0.        , 0.        , 0.        ],
        [0.81450625, 0.        , 0.857375  , 0.77378094],
        [0.81450625, 0.9025    , 0.9025    , 0.        ],
        [0.857375  , 0.95      , 0.        , 0.857375  ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.

In [ ]:
username = "elliemci"
repo_name = "q-FrozenLake-v1-4x4-noSlippery"
push_to_hub(
    repo_id=f"{username}/{repo_name}",
    model=model,
    env=env)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/834 [00:00<?, ?B/s]

replay.mp4:   0%|          | 0.00/31.2k [00:00<?, ?B/s]

q-learning.pkl:   0%|          | 0.00/915 [00:00<?, ?B/s]

results.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

... (4 outputs left out)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...8ae94918bb/q-learning.pkl: 100%|##########|   915B /   915B            

Model is pushed to the Hub. View model at:  https://huggingface.co/elliemci/q-FrozenLake-v1-4x4-noSlippery


In [ ]:
from huggingface_hub import upload_file

# manually upload the video to the model card preview
upload_file(
    path_or_fileobj="./videos/replay.mp4",
    path_in_repo="replay.mp4",
    repo_id="elliemci/q-FrozenLake-v1-4x4-noSlippery",
    repo_type="model",
)

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/elliemci/q-FrozenLake-v1-4x4-noSlippery/commit/3468b4a9ee75cbd03189245747d00e81fe2a09a3', commit_message='Upload replay.mp4 with huggingface_hub', commit_description='', oid='3468b4a9ee75cbd03189245747d00e81fe2a09a3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/elliemci/q-FrozenLake-v1-4x4-noSlippery', endpoint='https://huggingface.co', repo_type='model', repo_id='elliemci/q-FrozenLake-v1-4x4-noSlippery'), pr_revision=None, pr_num=None)

## <font size="20">**Taxi-v3**</font>

### Environment

In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array", is_rainy=False, fickle_passenger=False)
#print(dir(env))

### Observation Space
There are 500 discrete states since there are 25 taxi positions, 5 possible locations of the passenger, and 4 destination locations:

Passenger locations:

* 0: Red

* 1: Green

* 2: Yellow

* 3: Blue

* 4: In taxi

Destinations:

* 0: Red

* 1: Green

* 2: Yellow

* 3: Blue

Observation is returned as an int() that encodes the corresponding state, calculated by ((taxi_row * 5 + taxi_col) * 5 + passenger_location) * 4 + destination

In [ ]:
state_space = env.observation_space.n
print("There are ", state_space, " possible states")

There are  500  possible states


In [ ]:
print(dir(env))

['__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_action_space', '_cached_spec', '_elapsed_steps', '_max_episode_steps', '_metadata', '_np_random', '_np_random_seed', '_observation_space', '_saved_kwargs', 'action_space', 'class_name', 'close', 'env', 'get_wrapper_attr', 'has_wrapper_attr', 'metadata', 'np_random', 'np_random_seed', 'observation_space', 'render', 'render_mode', 'reset', 'set_wrapper_attr', 'spec', 'step', 'unwrapped', 'wrapper_spec']


### Action Space

* 0: move south
* 1: move north
* 2: move east
* 3: move west
* 4: pickup passenger
* 5: drop off passenger


In [ ]:
dir(env.action_space)

['__annotations__',
 '__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_np_random',
 '_shape',
 'contains',
 'dtype',
 'from_jsonable',
 'is_np_flattenable',
 'n',
 'np_random',
 'sample',
 'seed',
 'shape',
 'start',
 'to_jsonable']

In [ ]:
action_space = env.action_space.n
print("There are ", action_space, " possible actions")

There are  6  possible actions


### Rewards

* -1 per step unless other reward is triggered.

* +20 delivering passenger.

* -10 executing “pickup” and “drop-off” actions illegally, like moving into a wall; info["action_mask"] contains a np.ndarray for each of the actions specifying if the action will change the state.

### Q-Table

In [ ]:
Qtable_taxi = initialize_q_table(state_space, action_space)

print("Qtable shape: ", Qtable_taxi.shape)
print(Qtable_taxi)

Qtable shape:  (500, 6)
[[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]


### Hyperparameters

In [ ]:
# training parameters
n_training_episodes = 25000   # training episodes
learning_rate = 0.7           # learning rate

# evaluation parameters
n_eval_episodes = 100         # test episodes

# evaluation seed array allows to evaluate the agent with the same taxi starting position
eval_seed = [16,54,165,177,191,191,120,80,149,178,48,38,6,125,174,73,50,172,100,148,146,6,25,40,68,148,49,167,9,97,164,176,61,7,54,55,
 161,131,184,51,170,12,120,113,95,126,51,98,36,135,54,82,45,95,89,59,95,124,9,113,58,85,51,134,121,169,105,21,30,11,50,65,12,43,82,145,152,97,106,55,31,85,38,
 112,102,168,123,97,21,83,158,26,80,63,5,81,32,11,28,148]

# environment parameters
env_id = "Taxi-v3"           # environment name
max_steps = 99               # max steps per episode
gamma = 0.95                 # discounting rate

# exploration parameters
max_epsilon = 1.0             # exploration probability at start
min_epsilon = 0.05            # min exploration probability
decay_rate = 0.0005            # exponential decay rate for exploration prob

### Training

In [ ]:
Qtable_taxi = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_taxi)
Qtable_taxi

  0%|          | 0/25000 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
 [ 0.53368333  2.75200369  1.61440351  2.75200369 -7.38559649 -7.38559649]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [ 1.61440351  2.75200369  2.75200369  3.94947757 -6.24799631 -6.24799631]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [ 2.75200369  3.94947757  2.75200369  3.94947757  5.20997639 -5.05052243]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [ 6.53681725  5.20997639  6.53681725  5.20997639 -3.79002361  3.94947757]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [ 7.93349184  5.20997639  7.93349184  6.53681725 -2.46318275 -2.46318275]
The row of Qtable that corresponds to the current state, containing the q-values for all p

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 2.75200369,  3.94947757,  2.75200369,  3.94947757,  5.20997639,
        -5.05052243],
       [ 7.93349184,  9.40367562,  7.93349184,  9.40367562, 10.9512375 ,
         0.40367562],
       ...,
       [10.95123744, 12.58025   , 10.95121432,  9.40367562,  1.95122274,
         1.94915189],
       [ 5.20317079,  6.53681725,  4.32373442,  6.53666647, -3.79720475,
        -3.79329841],
       [16.1       , 14.295     , 16.1       , 18.        ,  7.1       ,
         7.1       ]])

## Evaluation

In [ ]:
# evaluate Q-learning agent
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_taxi, eval_seed)

  0%|          | 0/100 [00:00<?, ?it/s]

The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [ -2.40248269  -2.42011795  -3.43478068  -0.49300084 -12.37106851
 -11.08173032]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [-1.46835079 -1.46835079 -1.4683508   0.53368333 -9.49300084 -9.49300084]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [-0.49300084 -0.49300084 -0.49300084  1.61440351 -8.46631667 -8.46631667]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [ 0.53368333  2.75200369  0.53368333  2.75200369 -7.38559649 -7.38559649]
The row of Qtable that corresponds to the current state, containing the q-values for all possible actions in that state:
 [ 1.61440351  3.94947757  2.75200369  3.94947757 -6.24799631 -6.24799631]
The row of Qt

In [ ]:
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

Mean_reward=7.56 +/- 2.71


Push trained model to Hub

Create a model dictionary that contains the hyperparameters and Q-table

In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_taxi
}

In [ ]:
record_video(env, model["qtable"], "videos/replay.mp4")

In [ ]:
show_videos("videos", prefix="taxi-")

In [ ]:
username = "elliemci"
repo_name = "q-Taxi-v3-noRainy"

push_to_hub(
    repo_id=f"{username}/{repo_name}",
    model=model,
    env=env
)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

q-learning.pkl:   0%|          | 0.00/24.6k [00:00<?, ?B/s]

replay.mp4:   0%|          | 0.00/119k [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/730 [00:00<?, ?B/s]

results.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

... (6 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...2d53df9659f64a/replay.mp4: 100%|##########|  104kB /  104kB            

  ...df9659f64a/q-learning.pkl: 100%|##########| 24.6kB / 24.6kB            

Model is pushed to the Hub. View model at:  https://huggingface.co/elliemci/q-Taxi-v3-noRainy


In [ ]:
from huggingface_hub import upload_file

# manually upload the video to the model card preview
upload_file(
    path_or_fileobj="./videos/taxi-replay.mp4",
    path_in_repo="replay.mp4",
    repo_id="elliemci/q-Taxi-v3-noRainy",
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ./videos/taxi-replay.mp4    : 100%|##########|  117kB /  117kB            

CommitInfo(commit_url='https://huggingface.co/elliemci/q-Taxi-v3-noRainy/commit/4130b7af560964fe56e4cdf82c95f2ee4258248a', commit_message='Upload replay.mp4 with huggingface_hub', commit_description='', oid='4130b7af560964fe56e4cdf82c95f2ee4258248a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/elliemci/q-Taxi-v3-noRainy', endpoint='https://huggingface.co', repo_type='model', repo_id='elliemci/q-Taxi-v3-noRainy'), pr_revision=None, pr_num=None)

## Load from Hub

In [ ]:
from urllib.error import HTTPError

from huggingface_hub import hf_hub_download


def load_from_hub(repo_id: str, filename: str) -> str:
    """
    Download a model from Hugging Face Hub.
    :param repo_id: id of the model repository from the Hugging Face Hub
    :param filename: name of the model zip file from the repository
    """
    # Get the model from the Hub, download and cache the model on your local disk
    pickle_model = hf_hub_download(
        repo_id=repo_id,
        filename=filename
    )

    with open(pickle_model, 'rb') as f:
      downloaded_model_file = pickle.load(f)

    return downloaded_model_file

In [ ]:
# try another model
model = load_from_hub(repo_id="ThomasSimonini/q-Taxi-v3", filename="q-learning.pkl") # Try to use another model

print(model)
env = gym.make(model["env_id"])

evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

In [ ]:
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

Mean_reward=7.56 +/- 2.71


In [ ]:
model

{'env_id': 'Taxi-v3',
 'max_steps': 99,
 'n_training_episodes': 25000,
 'n_eval_episodes': 100,
 'eval_seed': [16,
  54,
  165,
  177,
  191,
  191,
  120,
  80,
  149,
  178,
  48,
  38,
  6,
  125,
  174,
  73,
  50,
  172,
  100,
  148,
  146,
  6,
  25,
  40,
  68,
  148,
  49,
  167,
  9,
  97,
  164,
  176,
  61,
  7,
  54,
  55,
  161,
  131,
  184,
  51,
  170,
  12,
  120,
  113,
  95,
  126,
  51,
  98,
  36,
  135,
  54,
  82,
  45,
  95,
  89,
  59,
  95,
  124,
  9,
  113,
  58,
  85,
  51,
  134,
  121,
  169,
  105,
  21,
  30,
  11,
  50,
  65,
  12,
  43,
  82,
  145,
  152,
  97,
  106,
  55,
  31,
  85,
  38,
  112,
  102,
  168,
  123,
  97,
  21,
  83,
  158,
  26,
  80,
  63,
  5,
  81,
  32,
  11,
  28,
  148],
 'learning_rate': 0.7,
 'gamma': 0.95,
 'epsilon': 1.0,
 'max_epsilon': 1.0,
 'min_epsilon': 0.05,
 'decay_rate': 0.005,
 'qtable': array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ],
        [ 2.75200369,  3.949